# BÜYÜK VERİ FİNAL PROJESİ

## PySpark ve LDA Kullanılarak Haber Verileri Üzerinde Konu Modelleme

### İsim:
Egemen Gündüz

### Numara:
25281907

### Proje Konusu:
Haber Verileri ile Konu Modelleme

### Kullanılan Teknolojiler:
Apache Spark, PySpark, Spark MLlib, Hadoop HDFS, Docker, Jupyter Notebook

# Özet

# İçindekiler

1. Giriş
2. Problem Tanımı
3. Literatür
4. Veri Seti
5. Yöntem
6. Büyük Veri İşleme Süreci
7. Deneysel Sonuçlar
8. Tartışma
9. Sonuç
10. Kaynakça

# 1. Giriş

# 2. Problem Tanımı

# 3. Literatür

# 4. Veri Seti

Bu projede Kaggle platformunda yer alan “News Category Dataset” veri seti kullanılmıştır.

**Veri seti bağlantısı**:

https://www.kaggle.com/datasets/rmisra/news-category-dataset

Veri seti yukarıdaki linkten zip formatında indirilir ve extract edilir. "News_Category_Dataset_v3.json" isimli dosya notebook ile aynı dizine getirilir. **Bu işlem ön kontrol için yapılıyor, daha sonra dağıtık depolama ve işleme yapılacaktır.**

Veri seti; haber başlıkları, kısa açıklamalar, kategori bilgileri ve tarih verilerinden oluşmaktadır. Veri seti büyük ölçekli metin verisi içerdiğinden dolayı Apache Spark ile dağıtık veri işleme süreçleri için uygun yapıdadır.

In [1]:
import pandas as pd

In [2]:
df = pd.read_json(
    "News_Category_Dataset_v3.json",
    lines=True
)

In [3]:
df.head()

,link,headline,category,short_description,authors,date
0,https://www.huffpost.com/entry/covid-boosters-...,Over 4 Million Americans Roll Up Sleeves For O...,U.S. NEWS,Health experts said it is too early to predict...,"Carla K. Johnson, AP",2022-09-23
1,https://www.huffpost.com/entry/american-airlin...,"American Airlines Flyer Charged, Banned For Li...",U.S. NEWS,He was subdued by passengers and crew when he ...,Mary Papenfuss,2022-09-23
2,https://www.huffpost.com/entry/funniest-tweets...,23 Of The Funniest Tweets About Cats And Dogs ...,COMEDY,"""Until you have a dog you don't understand wha...",Elyse Wanshel,2022-09-23
3,https://www.huffpost.com/entry/funniest-parent...,The Funniest Tweets From Parents This Week (Se...,PARENTING,"""Accidentally put grown-up toothpaste on my to...",Caroline Bologna,2022-09-23
4,https://www.huffpost.com/entry/amy-cooper-lose...,Woman Who Called Cops On Black Bird-Watcher Lo...,U.S. NEWS,Amy Cooper accused investment firm Franklin Te...,Nina Golgowski,2022-09-22


# 5. Yöntem

## 5.1 Kullanılan Yöntemler
## 5.2 TF-IDF Yaklaşımı
## 5.3 LDA Algoritması

# 6. Büyük Veri İşleme Süreci

## 6.1 SparkSession Oluşturulması
Bu aşamada PySpark kullanılarak SparkSession oluşturulmuştur. SparkSession, Spark uygulamasının başlangıç noktasıdır ve DataFrame işlemleri, SQL işlemleri ve Spark MLlib süreçleri için temel arayüz sağlar.


In [4]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lower, regexp_replace, concat_ws, trim, length

In [7]:
spark = SparkSession.builder \
    .appName("HaberAnalizi") \
    .master("spark://localhost:7077") \
    .config("spark.driver.host", "172.18.0.1") \
    .config("spark.driver.bindAddress", "0.0.0.0") \
    .config("spark.driver.port", "4045") \
    .config("spark.blockManager.port", "4046") \
    .config("spark.hadoop.fs.defaultFS", "hdfs://172.18.0.4:9000") \
    .config("spark.hadoop.dfs.client.use.datanode.hostname", "false") \
    .config("spark.executor.memory", "512m") \
    .config("spark.cores.max", "2") \
    .config("spark.executor.extraJavaOptions", "-Dhadoop.security.logger=ERROR,RFAS") \
    .getOrCreate()

sc = spark.sparkContext

print("Spark Master:", sc.master)
print("Default Parallelism:", sc.defaultParallelism)

Spark Master: spark://localhost:7077
Default Parallelism: 2


26/05/20 02:39:04 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [9]:
data_path = "hdfs://172.18.0.4:9000/bigdata/news/input/News_Category_Dataset_v3.json"

df = spark.read.json(data_path)

print("Toplam kayıt:", df.count())
print("Partition sayısı:", df.rdd.getNumPartitions())

Toplam kayıt: 209527
Partition sayısı: 2


## 6.2 Veri Setinin PySpark ile Okunması

News Category Dataset JSON formatında PySpark DataFrame olarak okunmuştur. Bu işlem pandas yerine Spark kullanılarak gerçekleştirilmiştir.

In [10]:
df.limit(5).toPandas()

,authors,category,date,headline,link,short_description
0,"Carla K. Johnson, AP",U.S. NEWS,2022-09-23,Over 4 Million Americans Roll Up Sleeves For O...,https://www.huffpost.com/entry/covid-boosters-...,Health experts said it is too early to predict...
1,Mary Papenfuss,U.S. NEWS,2022-09-23,"American Airlines Flyer Charged, Banned For Li...",https://www.huffpost.com/entry/american-airlin...,He was subdued by passengers and crew when he ...
2,Elyse Wanshel,COMEDY,2022-09-23,23 Of The Funniest Tweets About Cats And Dogs ...,https://www.huffpost.com/entry/funniest-tweets...,"""Until you have a dog you don't understand wha..."
3,Caroline Bologna,PARENTING,2022-09-23,The Funniest Tweets From Parents This Week (Se...,https://www.huffpost.com/entry/funniest-parent...,"""Accidentally put grown-up toothpaste on my to..."
4,Nina Golgowski,U.S. NEWS,2022-09-22,Woman Who Called Cops On Black Bird-Watcher Lo...,https://www.huffpost.com/entry/amy-cooper-lose...,Amy Cooper accused investment firm Franklin Te...


26/05/20 02:52:06 ERROR TaskSchedulerImpl: Lost executor 1 on 172.18.0.10: Worker shutting down
26/05/20 02:52:09 ERROR TaskSchedulerImpl: Lost executor 0 on 172.18.0.9: Remote RPC client disassociated. Likely due to containers exceeding thresholds, or network issues. Check driver logs for WARN messages.
26/05/20 02:52:14 WARN StandaloneAppClient$ClientEndpoint: Connection to 0.0.0.0:7077 failed; waiting for master to reconnect...
26/05/20 02:52:14 WARN StandaloneSchedulerBackend: Disconnected from Spark cluster! Waiting for reconnection...


In [ ]:
from pyspark.sql.functions import col

category_counts = df.groupBy("category") \
    .count() \
    .orderBy(col("count").desc())

category_counts.limit(20).toPandas()

In [ ]:
df_selected = df.select(
    "category",
    "headline",
    "short_description",
    "date"
)

df_selected.limit(5).toPandas()

## 6.3 Veri Ön İşleme
## 6.4 Tokenization
## 6.5 Stopwords Temizleme
## 6.6 TF-IDF Özellik Çıkarımı
## 6.7 LDA Modelinin Eğitilmesi
## 6.8 DataFrame İşlemleri: filter, groupBy ve aggregation

# 7. Deneysel Sonuçlar

## 7.1 Kategori Dağılımı
## 7.2 Konular ve Anahtar Kelimeler
## 7.3 Konu Dağılımları

# 8. Tartışma

# 9. Sonuç

# 10. Kaynakça